In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.gold.dim_clientes AS
SELECT customer_id, name, email, phone, state, created_at
FROM ecommerce.silver.customers;

In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.gold.dim_produtos AS
SELECT product_id, product_name, category, price, is_active
FROM ecommerce.silver.products;

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.gold.fato_pedidos AS
SELECT
  o.order_id,
  o.customer_id,
  o.order_date,
  o.order_day,
  o.status,
  o.payment_method,
  o.installments,
  o.qtd_itens,
  ROUND(COALESCE(i.valor_total, 0), 2) AS valor_total,
  c.customer_id IS NULL                AS cliente_nao_cadastrado
FROM ecommerce.silver.orders o
LEFT JOIN (
  SELECT order_id, SUM(total_item) AS valor_total
  FROM ecommerce.silver.order_items
  GROUP BY order_id
) i ON i.order_id = o.order_id
LEFT JOIN ecommerce.silver.customers c ON c.customer_id = o.customer_id;

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.gold.fato_itens AS
SELECT
  i.order_id,
  o.order_day,
  o.status,
  i.product_id,
  i.quantity,
  i.unit_price,
  i.total_item,
  p.product_id IS NULL AS produto_nao_cadastrado
FROM ecommerce.silver.order_items i
JOIN ecommerce.silver.orders o ON o.order_id = i.order_id
LEFT JOIN ecommerce.silver.products p ON p.product_id = i.product_id;

In [0]:
%sql
CREATE OR REPLACE VIEW ecommerce.gold.fato_eventos AS
SELECT event_id, session_id, customer_id, event_type, product_id, platform, event_timestamp
FROM ecommerce.silver.events;

In [0]:
%sql
-- VENDAS DIÁRIAS
CREATE OR REPLACE TABLE ecommerce.gold.vendas_diarias AS
SELECT
  o.order_day,
  COUNT(*)                                             AS qtd_pedidos,
  COUNT_IF(o.status = 'completed')                     AS pedidos_concluidos,
  COUNT_IF(o.status = 'canceled')                      AS pedidos_cancelados,
  ROUND(COUNT_IF(o.status = 'canceled') / COUNT(*), 4) AS taxa_cancelamento,
  ROUND(SUM(CASE WHEN o.status = 'completed' THEN i.total_item END), 2) AS receita,
  ROUND(SUM(CASE WHEN o.status = 'completed' THEN i.total_item END)
        / NULLIF(COUNT_IF(o.status = 'completed'), 0), 2) AS ticket_medio
FROM ecommerce.silver.orders o
LEFT JOIN (
  SELECT order_id, SUM(total_item) AS total_item
  FROM ecommerce.silver.order_items
  GROUP BY order_id
) i ON i.order_id = o.order_id
GROUP BY o.order_day;

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce.gold.obt_ecommerce AS

-- 1) VENDAS: uma linha por item de pedido
SELECT
  'venda'                                              AS tipo_registro,
  o.order_day                                          AS dia,
  i.order_id,
  o.order_date,
  o.status,
  o.payment_method,
  o.installments,
  o.customer_id,
  c.name                                               AS customer_name,
  c.state                                              AS customer_state,
  o.cliente_nao_cadastrado,
  i.product_id,
  COALESCE(p.product_name, 'Produto não cadastrado')   AS product_name,
  COALESCE(p.category, 'Não informado')                AS category,
  i.produto_nao_cadastrado,
  i.quantity,
  i.unit_price,
  i.total_item,
  CAST(NULL AS STRING) AS platform,
  CAST(NULL AS STRING) AS event_type,
  CAST(NULL AS BIGINT) AS qtd_eventos,
  CAST(NULL AS BIGINT) AS qtd_sessoes
FROM ecommerce.gold.fato_itens i
JOIN ecommerce.gold.fato_pedidos o ON o.order_id = i.order_id
LEFT JOIN ecommerce.gold.dim_clientes c ON c.customer_id = o.customer_id
LEFT JOIN ecommerce.gold.dim_produtos p ON p.product_id = i.product_id

UNION ALL

-- 2) EVENTOS: agregados por dia, plataforma e tipo de evento
SELECT
  'evento'                          AS tipo_registro,
  CAST(e.event_timestamp AS DATE)   AS dia,
  CAST(NULL AS STRING)              AS order_id,
  CAST(NULL AS TIMESTAMP)           AS order_date,
  CAST(NULL AS STRING)              AS status,
  CAST(NULL AS STRING)              AS payment_method,
  CAST(NULL AS BIGINT)              AS installments,
  CAST(NULL AS BIGINT)              AS customer_id,
  CAST(NULL AS STRING)              AS customer_name,
  CAST(NULL AS STRING)              AS customer_state,
  CAST(NULL AS BOOLEAN)             AS cliente_nao_cadastrado,
  CAST(NULL AS BIGINT)              AS product_id,
  CAST(NULL AS STRING)              AS product_name,
  CAST(NULL AS STRING)              AS category,
  CAST(NULL AS BOOLEAN)             AS produto_nao_cadastrado,
  CAST(NULL AS BIGINT)              AS quantity,
  CAST(NULL AS DOUBLE)              AS unit_price,
  CAST(NULL AS DOUBLE)              AS total_item,
  e.platform,
  e.event_type,
  COUNT(*)                          AS qtd_eventos,
  COUNT(DISTINCT e.session_id)      AS qtd_sessoes
FROM ecommerce.gold.fato_eventos e
GROUP BY CAST(e.event_timestamp AS DATE), e.platform, e.event_type;

In [0]:
%sql
SELECT 'fato_pedidos' AS objeto, COUNT(*) AS linhas FROM ecommerce.gold.fato_pedidos
UNION ALL SELECT 'fato_itens',      COUNT(*) FROM ecommerce.gold.fato_itens
UNION ALL SELECT 'dim_clientes',    COUNT(*) FROM ecommerce.gold.dim_clientes
UNION ALL SELECT 'dim_produtos',    COUNT(*) FROM ecommerce.gold.dim_produtos
UNION ALL SELECT 'vendas_diarias',  COUNT(*) FROM ecommerce.gold.vendas_diarias;